In [26]:
%run "./dq_framework"

StatementMeta(, 93700e91-bfae-4ea2-85dd-f8caf5b5b9be, 42, Finished, Available, Finished, True)

In [27]:
%run "./log_monitoring"

StatementMeta(, 93700e91-bfae-4ea2-85dd-f8caf5b5b9be, 44, Finished, Available, Finished, True)

In [28]:
from pyspark.sql.functions import col, udf, lit, when, isnan, monotonically_increasing_id,\
    input_file_name, regexp_extract, to_timestamp, current_timestamp,\
    col, lit, isnan, collect_list, concat_ws, col, max
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, DataType, TimestampType
from pyspark.sql.types import BooleanType
import re
import uuid

StatementMeta(, 93700e91-bfae-4ea2-85dd-f8caf5b5b9be, 45, Finished, Available, Finished, False)

In [29]:
system=''
table_name=''
cleanSchema=''
watermarkColumn=''
watermarkValue=''
loadType=''
keyColumn=''
lakehouse=''
source_schema=''
target_schema=''
config_schema=''
config_table=''
dq_table=''
monitoring_table=''

StatementMeta(, 93700e91-bfae-4ea2-85dd-f8caf5b5b9be, 46, Finished, Available, Finished, False)

In [30]:
rawtable=f"{lakehouse}.{source_schema}.{system}_{table_name}"
cleantable=f"{lakehouse}.{target_schema}.{system}_{table_name}"
qurantine_cleantable=f"{lakehouse}.{target_schema}.{system}_quarantine_{table_name}"
monitoring_table_fullname=f"{lakehouse}.{config_schema}.{monitoring_table}"

StatementMeta(, 93700e91-bfae-4ea2-85dd-f8caf5b5b9be, 47, Finished, Available, Finished, False)

In [31]:
def full_load():
    source_raw_count=0
    target_raw_count=0
    dq_issue_count=0
    remarks=''
    if loadType=='Full':
        if spark.catalog.tableExists(cleantable):
            max_sys_file_timestamp=spark.sql(f"SELECT max(sys_file_timestamp) FROM {cleantable}").first()[0]

            if max_sys_file_timestamp is None:
                max_sys_file_timestamp=spark.sql(f"SELECT max(sys_file_timestamp) FROM {rawtable}").first()[0]
                df=spark.sql(f"SELECT * FROM {rawtable} where sys_file_timestamp>='{max_sys_file_timestamp}'")
                 #######  Run dq_check ######
                log_df,invalid_df,df = fn_execute_dq(df,lakehouse,config_schema,dq_table,table_name,"clean")

                transformed_df=(eval(f"(df.{cleanSchema})"))
                transformed_df.write.formate("delta").mode("overwrite").saveAsTable(cleantable)
                invalid_df.write.format("delta").mode("append").saveAsTable(quarantine_cleantable)

                source_raw_count=df.count()
                target_raw_count=transformed_df.count()
                dq_issue_count=invalid_df.count()
                remarks=f"Data loaded to {cleantable} table"

            else:
                df=spark.sql(f"SELECT * FROM {rawtable} where sys_file_timestamp>='{max_sys_file_timestamp}'")
                if df.rdd.isEmpty():
                    print("No new data to load")
                    remarks='No new data to load'
                else:
                    ######## run dq check #########
                    log_df,invalid_df,df = fn_execute_dq(df,lakehouse,config_schema,dq_table,table_name,"clean")
                    
                    
                    transformed_df=(eval(f"(df.{cleanSchema})"))
                    transformed_df.write.formate("delta").mode("overwrite").saveAsTable(cleantable)
                    invalid_df.write.format("delta").mode("append").saveAsTable(quarantine_cleantable)

                    source_raw_count=df.count()
                    target_raw_count=transformed_df.count()
                    dq_issue_count=invalid_df.count()
                    remarks=f"Data loaded to {cleantable} table"
        else:
            df=spark.sql(f"SELECT * FROM {rawtable}")
            ###### ru dq check ####
            log_df,invalid_df,df = fn_execute_dq(df,lakehouse,config_schema,dq_table,table_name,"clean")
                    
            transformed_df=(eval(f"(df.{cleanSchema})"))
            transformed_df.write.formate("delta").mode("overwrite").saveAsTable(cleantable)
            invalid_df.write.format("delta").mode("append").saveAsTable(quarantine_cleantable)

            source_raw_count=df.count()
            target_raw_count=transformed_df.count()
            dq_issue_count=invalid_df.count()
            remarks=f"Data loaded to {cleantable} table"
    return source_raw_count,target_raw_count,dq_issue_count,remarks



StatementMeta(, 93700e91-bfae-4ea2-85dd-f8caf5b5b9be, 48, Finished, Available, Finished, False)

In [32]:
def generate_merge():
    schema_info=spark.sql(f"DESCRIBE {rawtable}")

    #sourcequery=f"select * from {rawtable} where {watermarkColumn}>'{watermarkValue}'"
    sourceview=f"view_{system}_{table_name}_temp"

    key_column_array=keyColumn.split(",")
    join_condition = " AND ".join([f"Target.{col}" + f" = Source.{col}" for col in key_column_array])

    #### Merge Script #####
    update_clause=''
    insert_cloase_target=''
    insert_cloase_source='' 

    for row in schema_info.collect():
        if row[col_name] not in key_column_array:
            update_clause += f" Target.{row['col_name']} = Source.{row['col_name']},\n"
        
        insert_cloase_source += f"Source.{row['col_name']}, "
        insert_cloase_target += f"{row['col_name']}, "

    update_clause=update_clause.rstrip(",\n")
    insert_cloase_source=insert_cloase_source.rstrip(", ")
    insert_cloase_target=insert_cloase_target.rstrip(", ")

    merge_script=''
    merge_script= f"merge into {cleantable} as Target \n \
                    using {sourceview} as Source \n \
                    on {join_condition}                 \n \
                    when matched then                   \n \
                    update set {update_clause}          \n \
                    when not matched then               \n \
                    insert ({insert_cloase_target}) values({insert_cloase_source});"

    spark.sql(f"""INSERT INTO {lakehouse}.{config_schema}.{config_table}(system,tier,table_name,key_name,value) 
                  VALUES('{system}','{target_schema}','{table_name}',"mergeScript",'{merge_script}')""")

StatementMeta(, 93700e91-bfae-4ea2-85dd-f8caf5b5b9be, 49, Finished, Available, Finished, False)

In [33]:
def incremental_load():
    source_raw_count=0
    target_raw_count=0
    dq_issue_count=0
    remarks=''

    if loadType=='Incremental':

        sourcequery=spark.sql(f"""select * from (select *,
                                                    raw_number() over(partition by {keyColumn}
                                                                        order by sys_insert_timestamp desc) as rn
                                                from {rawtable} where {watermarkColumn}>'{watermarkValue}'
                                                ) where rn=1""").drop("rn")

    if sourcequery.rdd.isEmpty():
        print("No new data to load")
        remarks="No new data to load"

    else:
        ##### run dq_check #####
        log_df,invalid_df,df = fn_execute_dq(sourcequery,lakehouse,config_schema,dq_table,table_name,"clean")

        sourceview=f"view_{system}_{table_name}_temp"
        df.createOrReplaceTempview(sourceview)

        source_raw_count=sourcequery.count()
        target_raw_count=df.count()
        dq_issue_count=invalid_df.count()
        remarks=f"Data loaded to {cleantable} table"
    
        if spark.catalog.tableExists(cleantable):
            merge_script=spark.sql(f"""SELECT VALUES from {lakehouse}.{config_schema}.{config_table}
                        where key_name='mergeScript' and table_name='{table_name}'
                        and tier='{target_schema}' and system='{system}'""").first()[0]
            print("merge start")
            spark.sql(merge_script)
            print("merge complete")

        else:

            transformed_df=(eval(f"(df.{cleanSchema})"))
            transformed_df.write.formate("delta").mode("overwrite").saveAsTable(cleantable)
            print("initial load done, table created")
            spark.sql(f"""DELETE from {lakehouse}.{config_schema}.{config_table}
                        where key_name='merge_script' and table_name='{table_name}'
                        and tier='{target_schema}' and system='{system}'""")

            generate_merge()
            print("merge script generated")
        
        invalid_df.write.format("delta").mode("append").saveAsTable(quarantine_cleantable)
        watermark_update()
    return  source_raw_count,target_raw_count,dq_issue_count,remarks

StatementMeta(, 93700e91-bfae-4ea2-85dd-f8caf5b5b9be, 50, Finished, Available, Finished, False)

In [34]:
def watermark_update():
    if loadType=='Incremental':
        df=spark.table(rawtable)

        max_value=df.select(max(col(watermarkColumn)).first()[0])

        if max_value is None:
            max_value='1900-01-01'

        max_value_formated_str = max_value

        update_query=f"""update {lakehouse}.{config_schema}.{config_table}
                                set value='{max_value_formated_str}'
                                where table_name='{table_name}'
                                and key_name='watermarkValue'
                                and tier='{target_schema}'"""
        
        spark.sql(update_query)
        print("watermark table updated")

StatementMeta(, 93700e91-bfae-4ea2-85dd-f8caf5b5b9be, 51, Finished, Available, Finished, False)

In [35]:
if spark.catalog.tableExists(rawtable):

    try:

        start_time = datetime.now()
        pipeline_name = "raw_to_clean"
        run_id = str(uuid.uuid4())
        layer_from = source_schema
        layer_to=target_schema

        log = start_pipeline_log(system, run_id, start_time, pipeline_name, layer_from, layer_to, f"{system}_{table_name}")
        
        if loadType=="Full":
            source_raw_count, target_raw_count, dq_issue_count, remarks = full_load()

        if loadType==" Incremental":
            source_raw_count, target_raw_count, dq_issue_count, remarks = incremental_load()
            
        end_pipeline_log(monitoring_table_fullname, log, "Success", source_raw_count, target_raw_count, dq_issue_count, remarks)
    except Exception as e:
        error_message=f"{e}"
        source_raw_count=0
        target_raw_count=0
        end_pipeline_log(monitoring_table_fullname,log,"Fail",source_raw_count,target_raw_count,0,'',error_message)

StatementMeta(, 93700e91-bfae-4ea2-85dd-f8caf5b5b9be, 52, Finished, Available, Finished, False)

ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near '.'.(line 1, pos 0)

== SQL ==
.._
^^^
